# Chapter 18 — GPU Optimizer & Gradient Clipping

> Course: **llm.c — Zero to Hero**, Chapter 18 of ~20.
> Builds on Chapter 8 (CPU AdamW), Chapter 13 (block reductions).

You wrote AdamW in C in Chapter 8. The GPU version is the same math, with two new wrinkles:

1. **Massively parallel update**. 124M parameters, each independent. One thread per parameter, grid-stride loop.
2. **Gradient clipping** requires the **global L2 norm** of all gradients — a *single scalar* aggregated across 124M floats spread over many tensors. That's a multi-kernel reduction.

This chapter covers both. By the end, you'll be able to read `llmc/adamw.cuh::adamw_kernel3` and `llmc/global_norm.cuh::global_norm_squared_kernel` and explain every line.

### Learning objectives

By the end of this chapter you will:

- Port the CPU AdamW from Chapter 8 to a GPU kernel.
- Compute the global L2 norm of `N` gradients via a **two-stage** reduction (block-wise sums, then aggregate).
- Apply **gradient clipping**: `g ← g * min(1, max_norm / current_norm)`.
- Explain why the clipping coefficient is computed *before* the AdamW step.


## 1. Concept — AdamW on GPU is Embarrassingly Parallel

Every parameter's AdamW update is independent of every other:

```
for each parameter i:
    m[i] = beta1 * m[i] + (1 - beta1) * g[i]
    v[i] = beta2 * v[i] + (1 - beta2) * g[i]^2
    param[i] -= lr * (m_hat / (sqrt(v_hat) + eps) + wd * param[i])
```

There's no cross-parameter communication. So a one-thread-per-parameter kernel does it:

```c
__global__ void adamw_kernel(float* param, float* m, float* v,
                             const floatX* grad, ...) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float g = (float) grad[i];
        m[i] = beta1 * m[i] + (1 - beta1) * g;
        v[i] = beta2 * v[i] + (1 - beta2) * g * g;
        // ... bias correction, update ...
    }
}
```

This is what `llmc/adamw.cuh::adamw_kernel3` is, plus mixed-precision casting (gradients in BF16, params in master FP32 + working BF16, m and v in FP32) and `Packed128` vectorized loads.


## 2. Concept — The Global Gradient Norm Problem

Gradient clipping computes:

$$\text{coef} = \min\!\left(1, \frac{\text{max\_norm}}{\sqrt{\sum_{i} g_i^2}}\right) \quad ; \quad g_i \leftarrow \text{coef} \cdot g_i$$

The sum is over **all 124M gradients across all 16 parameter tensors**. The whole gradient buffer is one contiguous block (Chapter 8), so we can treat it as one big array and reduce.

But: 124M elements is too big for one block (max 1024 threads). Solution: **two-stage reduction.**

```
Stage 1: each block reduces its slice of the array → one partial sum per block
         → write partial_sums[grid_size] (a few thousand floats)
Stage 2: a single small kernel sums those partial sums → one final scalar
```

This is the canonical pattern for any "reduce a giant array to one scalar" on GPU. `llm.c`'s `global_norm.cuh` has exactly this two-stage structure.


## 3. Demo — GPU AdamW vs PyTorch

In [ ]:
!mkdir -p course/ch18_build


In [ ]:
%%writefile course/ch18_build/adamw_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

__global__ void adamw_kernel(float* params, float* m, float* v,
                             const float* grads, size_t N,
                             float lr, float beta1, float beta2, float eps, float wd, int t) {
    size_t tid = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = (size_t)blockDim.x * gridDim.x;
    for (size_t i = tid; i < N; i += stride) {
        float p = params[i];
        float g = grads[i];
        float m_new = beta1 * m[i] + (1.0f - beta1) * g;
        float v_new = beta2 * v[i] + (1.0f - beta2) * g * g;
        m[i] = m_new; v[i] = v_new;
        float m_hat = m_new / (1.0f - powf(beta1, t));
        float v_hat = v_new / (1.0f - powf(beta2, t));
        params[i] = p - lr * (m_hat / (sqrtf(v_hat) + eps) + wd * p);
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    if (argc != 8) return 1;
    size_t N = (size_t) atoi(argv[1]);
    float lr=atof(argv[2]), b1=atof(argv[3]), b2=atof(argv[4]);
    float eps=atof(argv[5]), wd=atof(argv[6]); int t=atoi(argv[7]);

    float* h_params = (float*) rd("course/ch18_build/params.bin", N*4);
    float* h_grads  = (float*) rd("course/ch18_build/grads.bin",  N*4);
    float* h_m      = (float*) rd("course/ch18_build/m.bin",      N*4);
    float* h_v      = (float*) rd("course/ch18_build/v.bin",      N*4);

    float *d_p, *d_g, *d_m, *d_v;
    cudaMalloc(&d_p, N*4); cudaMalloc(&d_g, N*4); cudaMalloc(&d_m, N*4); cudaMalloc(&d_v, N*4);
    cudaMemcpy(d_p, h_params, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_g, h_grads,  N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_m, h_m,      N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_v, h_v,      N*4, cudaMemcpyHostToDevice);

    int block = 256;
    int grid  = 1024;
    adamw_kernel<<<grid, block>>>(d_p, d_m, d_v, d_g, N, lr, b1, b2, eps, wd, t);

    cudaMemcpy(h_params, d_p, N*4, cudaMemcpyDeviceToHost);
    cudaMemcpy(h_m,      d_m, N*4, cudaMemcpyDeviceToHost);
    cudaMemcpy(h_v,      d_v, N*4, cudaMemcpyDeviceToHost);

    FILE* f;
    f=fopen("course/ch18_build/params_after.bin","wb"); fwrite(h_params,4,N,f); fclose(f);
    f=fopen("course/ch18_build/m_after.bin","wb");      fwrite(h_m,     4,N,f); fclose(f);
    f=fopen("course/ch18_build/v_after.bin","wb");      fwrite(h_v,     4,N,f); fclose(f);
    cudaFree(d_p); cudaFree(d_g); cudaFree(d_m); cudaFree(d_v);
    free(h_params); free(h_grads); free(h_m); free(h_v);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18_build/adamw_gpu course/ch18_build/adamw_gpu.cu


In [ ]:
# Run our GPU AdamW step on a 64-param array, compare to torch.optim.AdamW (same math as Ch 8)
import numpy as np, torch, subprocess
torch.manual_seed(0); n = 64
params0 = torch.randn(n).float(); grads0 = torch.randn(n).float()
params0.numpy().tofile("course/ch18_build/params.bin")
grads0.numpy().tofile("course/ch18_build/grads.bin")
np.zeros(n, dtype=np.float32).tofile("course/ch18_build/m.bin")
np.zeros(n, dtype=np.float32).tofile("course/ch18_build/v.bin")
subprocess.run(["./course/ch18_build/adamw_gpu", str(n), "1e-3", "0.9", "0.999", "1e-8", "0.01", "1"], check=True)
params_c = np.fromfile("course/ch18_build/params_after.bin", dtype=np.float32)

p_t = params0.clone().requires_grad_(); p_t.grad = grads0.clone()
opt = torch.optim.AdamW([p_t], lr=1e-3, betas=(0.9,0.999), eps=1e-8, weight_decay=0.01)
opt.step()
err = np.max(np.abs(params_c - p_t.detach().numpy()))
print(f"GPU AdamW vs torch.optim.AdamW max diff: {err:.2e}")
print("PASS" if err < 1e-6 else "FAIL")


GPU and PyTorch's `torch.optim.AdamW` should agree to within float32 noise. The kernel is **15 lines of CUDA** for what is one of the most-iterated-over functions in deep learning.


## 4. Demo — Two-Stage Global Norm

Now let's compute the L2 norm of a giant array. We'll use a **block reduction** (Chapter 13) and a final aggregation kernel.


In [ ]:
%%writefile course/ch18_build/global_norm.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float v) {
    for (int o = 16; o > 0; o /= 2) v += __shfl_down_sync(0xffffffff, v, o);
    return v;
}
__device__ float blockReduceSum(float v) {
    __shared__ float ws[32];
    int lane = threadIdx.x & 31, wid = threadIdx.x >> 5;
    int nw = (blockDim.x + 31) / 32;
    v = warpReduceSum(v);
    if (lane == 0) ws[wid] = v;
    __syncthreads();
    v = (threadIdx.x < nw) ? ws[threadIdx.x] : 0.0f;
    if (wid == 0) v = warpReduceSum(v);
    return v;
}

// STAGE 1: each block computes the sum-of-squares of its slice, writes to partials[blockIdx.x]
__global__ void norm_squared_stage1(float* partials, const float* x, size_t N) {
    size_t tid    = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = (size_t)blockDim.x * gridDim.x;
    float local = 0.0f;
    for (size_t i = tid; i < N; i += stride) local += x[i] * x[i];
    float block_sum = blockReduceSum(local);
    if (threadIdx.x == 0) partials[blockIdx.x] = block_sum;
}

// STAGE 2: one block sums the (small) partials array
__global__ void norm_squared_stage2(float* out, const float* partials, int n_partials) {
    int t = threadIdx.x;
    float local = 0.0f;
    for (int i = t; i < n_partials; i += blockDim.x) local += partials[i];
    float total = blockReduceSum(local);
    if (t == 0) out[0] = total;
}

int main(void) {
    size_t N = 1u << 24;     // 16M floats
    float *h_x = (float*) malloc(N*4);
    double cpu_sumsq = 0.0;
    for (size_t i = 0; i < N; i++) {
        h_x[i] = (float)((i*7) % 13) / 30.0f - 0.2f;
        cpu_sumsq += (double)h_x[i] * h_x[i];
    }
    float *d_x, *d_partials, *d_out;
    int grid = 256;
    cudaMalloc(&d_x, N*4); cudaMalloc(&d_partials, grid*4); cudaMalloc(&d_out, 4);
    cudaMemcpy(d_x, h_x, N*4, cudaMemcpyHostToDevice);

    norm_squared_stage1<<<grid, 256>>>(d_partials, d_x, N);
    norm_squared_stage2<<<1, 256>>>(d_out, d_partials, grid);

    float gpu_sumsq;
    cudaMemcpy(&gpu_sumsq, d_out, 4, cudaMemcpyDeviceToHost);
    printf("CPU sum of squares: %.6f\n", cpu_sumsq);
    printf("GPU sum of squares: %.6f\n", gpu_sumsq);
    printf("rel diff: %.2e\n", fabs(gpu_sumsq - cpu_sumsq) / cpu_sumsq);
    printf("CPU L2 norm: %.6f   GPU L2 norm: %.6f\n", sqrt(cpu_sumsq), sqrtf(gpu_sumsq));

    cudaFree(d_x); cudaFree(d_partials); cudaFree(d_out);
    free(h_x);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18_build/global_norm course/ch18_build/global_norm.cu && ./course/ch18_build/global_norm


GPU and CPU norms should agree to ~`1e-5` relative — float32 sum order differs slightly between sequential and tree reductions. **You just reduced 16M floats to one scalar in 2 kernel launches.** Same pattern in `llmc/global_norm.cuh::global_norm_squared_kernel` and `global_norm_aggregate_kernel`, just with `Packed128` loads and BF16 input casting.


## 5. Concept — Putting It All Together: Clipped AdamW

The full training step on GPU:

```
1. Compute grad_norm² = Σ g_i² over all parameters    (two-stage reduction)
2. coef = min(1, max_norm / sqrt(grad_norm²))         (one scalar, computed on host or device)
3. Run adamw_kernel for each parameter, where the kernel multiplies grads by coef before using them
4. Cast master FP32 → BF16 working
```

`max_norm` is typically `1.0` for GPT-2 training. When grads explode, `coef < 1` shrinks them; otherwise `coef = 1` and AdamW runs normally.

This logic lives in `gpt2_update` in `train_gpt2.cu`. The clip happens **inside `adamw_kernel`**: each thread multiplies its `g` by `coef` *before* updating `m, v, params`. One kernel launch instead of a separate "scale gradients" pass — fusion at work again.


## 6. Translation Bridge

| PyTorch | GPU `llm.c` |
|---|---|
| `torch.optim.AdamW` | `adamw_kernel3` (one thread per param) |
| `torch.nn.utils.clip_grad_norm_(params, max_norm)` | `global_norm_squared_kernel` (stage 1) + `global_norm_aggregate_kernel` (stage 2) |
| Optimizer state (m, v) | `m_memory`, `v_memory` (each `num_parameters * 4` bytes, FP32) |
| `param.data` (the working copy) | `params_memory` (FP32 master, also there's a BF16 working copy in mixed precision builds) |


## 7. TODO Exercise — Combine Norm + Clip + AdamW

In [ ]:
%%writefile course/ch18_build/exercise1.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float v) { for (int o=16;o>0;o/=2) v += __shfl_down_sync(0xffffffff,v,o); return v; }
__device__ float blockReduceSum(float v) {
    __shared__ float ws[32];
    int lane = threadIdx.x & 31, wid = threadIdx.x >> 5;
    int nw = (blockDim.x + 31) / 32;
    v = warpReduceSum(v);
    if (lane == 0) ws[wid] = v;
    __syncthreads();
    v = (threadIdx.x < nw) ? ws[threadIdx.x] : 0.0f;
    if (wid == 0) v = warpReduceSum(v);
    return v;
}

// TODO: Compute grad_norm² with a single block (assume N <= block_size).
__global__ void compute_norm_sq(float* out, const float* g, int N) {
    int t = threadIdx.x;
    float v = 0;
    // TODO: load g[t]^2, blockReduceSum, write to out[0]
    if (t < N) v = g[t] * g[t];
    v = blockReduceSum(v);
    if (t == 0) out[0] = v;
}

// TODO: Apply gradient clipping in-place: g[i] *= min(1, max_norm / sqrt(grad_norm_sq[0]))
__global__ void clip_grads(float* g, const float* grad_norm_sq, float max_norm, int N) {
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < N) {
        // TODO: float current = sqrtf(grad_norm_sq[0]);
        // TODO: float coef = fminf(1.0f, max_norm / (current + 1e-6f));
        // TODO: g[t] *= coef;
    }
}

int main(void) {
    int N = 64;
    float h_g[64];
    for (int i = 0; i < N; i++) h_g[i] = 0.5f * (i + 1);   // norm² = sum(0.5(i+1))² ≈ huge
    float *d_g, *d_norm_sq;
    cudaMalloc(&d_g, N*4); cudaMalloc(&d_norm_sq, 4);
    cudaMemcpy(d_g, h_g, N*4, cudaMemcpyHostToDevice);

    compute_norm_sq<<<1, 64>>>(d_norm_sq, d_g, N);
    float h_norm_sq;
    cudaMemcpy(&h_norm_sq, d_norm_sq, 4, cudaMemcpyDeviceToHost);
    float current_norm = sqrtf(h_norm_sq);

    float max_norm = 1.0f;
    clip_grads<<<1, 64>>>(d_g, d_norm_sq, max_norm, N);
    cudaMemcpy(h_g, d_g, N*4, cudaMemcpyDeviceToHost);

    // After clipping the new norm should be ≈ max_norm
    float new_sumsq = 0; for (int i = 0; i < N; i++) new_sumsq += h_g[i] * h_g[i];
    float new_norm = sqrtf(new_sumsq);
    printf("original norm: %.4f   new norm after clip: %.4f   max_norm: %.4f\n",
           current_norm, new_norm, max_norm);
    printf("%s\n", fabsf(new_norm - max_norm) < 1e-3 ? "PASS" : "FAIL");
    cudaFree(d_g); cudaFree(d_norm_sq);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18_build/exercise1 course/ch18_build/exercise1.cu && ./course/ch18_build/exercise1


### Solution

In [ ]:
%%writefile course/ch18_build/exercise1_sol.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float v) { for (int o=16;o>0;o/=2) v += __shfl_down_sync(0xffffffff,v,o); return v; }
__device__ float blockReduceSum(float v) {
    __shared__ float ws[32];
    int lane = threadIdx.x & 31, wid = threadIdx.x >> 5;
    int nw = (blockDim.x + 31) / 32;
    v = warpReduceSum(v);
    if (lane == 0) ws[wid] = v;
    __syncthreads();
    v = (threadIdx.x < nw) ? ws[threadIdx.x] : 0.0f;
    if (wid == 0) v = warpReduceSum(v);
    return v;
}

__global__ void compute_norm_sq(float* out, const float* g, int N) {
    int t = threadIdx.x;
    float v = (t < N) ? g[t] * g[t] : 0.0f;
    v = blockReduceSum(v);
    if (t == 0) out[0] = v;
}

__global__ void clip_grads(float* g, const float* grad_norm_sq, float max_norm, int N) {
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < N) {
        float current = sqrtf(grad_norm_sq[0]);
        float coef = fminf(1.0f, max_norm / (current + 1e-6f));
        g[t] *= coef;
    }
}

int main(void) {
    int N = 64;
    float h_g[64];
    for (int i = 0; i < N; i++) h_g[i] = 0.5f * (i + 1);
    float *d_g, *d_norm_sq;
    cudaMalloc(&d_g, N*4); cudaMalloc(&d_norm_sq, 4);
    cudaMemcpy(d_g, h_g, N*4, cudaMemcpyHostToDevice);
    compute_norm_sq<<<1, 64>>>(d_norm_sq, d_g, N);
    float h_norm_sq; cudaMemcpy(&h_norm_sq, d_norm_sq, 4, cudaMemcpyDeviceToHost);
    float current_norm = sqrtf(h_norm_sq);
    float max_norm = 1.0f;
    clip_grads<<<1, 64>>>(d_g, d_norm_sq, max_norm, N);
    cudaMemcpy(h_g, d_g, N*4, cudaMemcpyDeviceToHost);
    float new_sumsq = 0; for (int i = 0; i < N; i++) new_sumsq += h_g[i] * h_g[i];
    float new_norm = sqrtf(new_sumsq);
    printf("original norm: %.4f   new norm: %.4f   max_norm: %.4f\n",
           current_norm, new_norm, max_norm);
    printf("%s\n", fabsf(new_norm - max_norm) < 1e-3 ? "PASS" : "FAIL");
    cudaFree(d_g); cudaFree(d_norm_sq);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18_build/exercise1_sol course/ch18_build/exercise1_sol.cu && ./course/ch18_build/exercise1_sol


## Recap

You now know:

- AdamW on GPU is **one thread per parameter**, grid-stride loop. 18 lines of CUDA.
- Global norm of all 124M grads = **two-stage reduction**: each block reduces a slice → partials array → one final block reduces the partials.
- Gradient clipping = scale grads by `min(1, max_norm/current_norm)` before AdamW. Fused inside `adamw_kernel` in production.
- The combined per-step pipeline: norm-stage1 → norm-stage2 → adamw → cast-to-BF16.

### What's next

**Chapter 19 — The Full Training Loop on GPU.** All the pieces in one place. We'll trace `gpt2_forward` in `train_gpt2.cu` and see exactly where each kernel lives, what gets cached, and how the per-step orchestration looks. The payoff: you understand the full GPU training loop end to end.

When you're ready, say **"proceed to Chapter 19"**.
